In [ ]:
!python -m pip install -U woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [ ]:
%pip install python-woc

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 3.0 MB/s eta 0:00:00
  Created wheel for python-lzf: filename=python_lzf-0.2.6-cp312-cp312-linux_x86_64.whl size=18589 sha256=aecc9dd7e9a5183adafc0205dd5c037ff0b1e92a9abdc461c8ea022bafafa7a0
  Stored in directory: /root/.cache/pip/wheels/0c/9b/11/c1baf6f3e2841c29980d34aaee5f98ac97de91ef429c068ef4
Successfully built python-lzf


In [ ]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
df_projects = pd.read_csv("net2prj.csv")

my_projects = df_projects[df_projects['netID'] == 'sydlwils']

projects = my_projects['WoC'].tolist()

print(projects)

list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

['parflow_parflow', 'inab_wes-backend', 'knossos-project_knossos', 'project-konda_pattern-based-quality-analysis', 'wistaria_materiappsinstaller', 'wtmatlock_flanker', 'openmd_openmd', 'google_uis-rnn', 'leftygray_cascade_calculations', 'nrel_openstudio-server']


,sha1,project
0,000b054f839a0eaba1d76e97da534b18df0e41f0,parflow_parflow


In [ ]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 10%|█         | 335/3194 [06:54<58:45,  1.23s/it]

Got Errors {'a253882c8d769235a4d3bdd0f6fd46c341364f03': 'Key a253882c8d769235a4d3bdd0f6fd46c341364f03 not found in /da5_fast/All.sha1c/commit_34.tch'}


 13%|█▎        | 430/3194 [08:52<56:48,  1.23s/it]

Got Errors {'ce5e25d66ea8226a41dba05ac0aa6900de9c9fba': 'Key ce5e25d66ea8226a41dba05ac0aa6900de9c9fba not found in /da5_fast/All.sha1c/commit_78.tch'}


 16%|█▌        | 499/3194 [10:17<55:27,  1.23s/it]

...

Got Errors {'8a4ff54799af1e8405b21ea4b498e56560c78257': 'Key 8a4ff54799af1e8405b21ea4b498e56560c78257 not found in /da5_fast/All.sha1c/commit_10.tch'}


 96%|█████████▋| 3077/3194 [1:03:31<02:24,  1.23s/it]

Got Errors {'ded9a39ac3ecee3489e98ada833517df1e8aab32': 'Key ded9a39ac3ecee3489e98ada833517df1e8aab32 not found in /da5_fast/All.sha1c/commit_94.tch'}


100%|██████████| 3194/3194 [1:05:55<00:00,  1.24s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,000b054f839a0eaba1d76e97da534b18df0e41f0,798c2f825ab29c72164485cd199f5a23e2d3029c,[fd6b0cde3513b99336aa9aa892c94bbbb11c8f82],George Artavanis <ga6@verde.princeton.edu>,1729186857,-0400,George Artavanis <ga6@verde.princeton.edu>,1729186857,-0400,Format with black\n,000b054f839a0eaba1d76e97da534b18df0e41f0,parflow_parflow
1,00254de2fe4f77d94ec1b40fe530e01c8cde7804,9cb5f8fc67721d1d6c1794e385cdb577ba2d5df8,[8b7b1168d3eca31db5c77d8059337440e4df3184],kvrigor <kvrigor@users.noreply.github.com>,1732290753,+0100,kvrigor <kvrigor@users.noreply.github.com>,1732290753,+0100,CI: Compile vanilla ParFlow GCC on Ubuntu 24.04\n,00254de2fe4f77d94ec1b40fe530e01c8cde7804,parflow_parflow


In [ ]:
print("Original commits:", len(df))
print("Retrieved commit records:", len(df_commit_data))
print("Missing:", len(df) - len(df_commit_data))
df_commit_data.columns

Original commits: 31937
Retrieved commit records: 31900
Missing: 37


Index(['commit', 'tree', 'parent', 'author', 'author_time', 'author_tz',
       'committer', 'committer_time', 'committer_tz', 'message', 'sha1',
       'project'],
      dtype='object')

In [ ]:
project_summary = (
    df_commit_data
    .groupby('project')
    .agg(
        num_commits=('commit', 'count'),
        num_authors=('author', 'nunique'),
        min_time=('committer_time', 'min'),
        max_time=('committer_time', 'max')
    )
    .reset_index()
)

project_summary

,project,num_commits,num_authors,min_time,max_time
0,google_uis-rnn,199,10,1539291780,1727274796
1,inab_wes-backend,2109,25,1601907721,1762171993
2,knossos-project_knossos,5530,61,1326397084,1761820051
3,leftygray_cascade_calculations,1070,4,1428986446,1740008603
4,nrel_openstudio-server,8818,68,1371703553,1755617530
5,openmd_openmd,3661,54,1095999403,1761583987
6,parflow_parflow,5365,135,1212532785,1776455380
7,project-konda_pattern-based-quality-analysis,3540,16,1576055433,1762359705
8,wistaria_materiappsinstaller,1231,19,1384268877,1759372566
9,wtmatlock_flanker,377,9,1603277905,1734535203


In [ ]:
project_summary['min_time'] = pd.to_datetime(
    project_summary['min_time'],
    unit='s',
    utc=True
)

project_summary['max_time'] = pd.to_datetime(
    project_summary['max_time'],
    unit='s',
    utc=True
)

project_summary

,project,num_commits,num_authors,min_time,max_time
0,google_uis-rnn,199,10,2018-10-11 21:03:00+00:00,2024-09-25 14:33:16+00:00
1,inab_wes-backend,2109,25,2020-10-05 14:22:01+00:00,2025-11-03 12:13:13+00:00
2,knossos-project_knossos,5530,61,2012-01-12 19:38:04+00:00,2025-10-30 10:27:31+00:00
3,leftygray_cascade_calculations,1070,4,2015-04-14 04:40:46+00:00,2025-02-19 23:43:23+00:00
4,nrel_openstudio-server,8818,68,2013-06-20 04:45:53+00:00,2025-08-19 15:32:10+00:00
5,openmd_openmd,3661,54,2004-09-24 04:16:43+00:00,2025-10-27 16:53:07+00:00
6,parflow_parflow,5365,135,2008-06-03 22:39:45+00:00,2026-04-17 19:49:40+00:00
7,project-konda_pattern-based-quality-analysis,3540,16,2019-12-11 09:10:33+00:00,2025-11-05 16:21:45+00:00
8,wistaria_materiappsinstaller,1231,19,2013-11-12 15:07:57+00:00,2025-10-02 02:36:06+00:00
9,wtmatlock_flanker,377,9,2020-10-21 10:58:25+00:00,2024-12-18 15:20:03+00:00


In [ ]:
sydlwils_project_summary = project_summary
print(sydlwils_project_summary)
sydlwils_project_summary.to_csv('sydlwils_project_summary.csv', index=False)

                                        project  num_commits  num_authors  \
0                                google_uis-rnn          199           10   
1                              inab_wes-backend         2109           25   
2                       knossos-project_knossos         5530           61   
3                leftygray_cascade_calculations         1070            4   
4                        nrel_openstudio-server         8818           68   
5                                 openmd_openmd         3661           54   
6                               parflow_parflow         5365          135   
7  project-konda_pattern-based-quality-analysis         3540           16   
8                  wistaria_materiappsinstaller         1231           19   
9                             wtmatlock_flanker          377            9   

                   min_time                  max_time  
0 2018-10-11 21:03:00+00:00 2024-09-25 14:33:16+00:00  
1 2020-10-05 14:22:01+00:00 2025-11-03 1

[number of stars, number of forks, last commit date]

https://github.com/parflow/parflow

Stars: 214

Forks: 130

Last Commit Date: 09/15/26


https://github.com/inab/wes-backend

Stars: 21

Forks: 5

Last Commit Date: 09/07/26

https://github.com/knossos-project/knossos

Stars: 77

Forks: 21

Last Commit Date: 05/22/26

https://github.com/project-konda/pattern-based-quality-analysis

Stars: 5

Forks: 3

Last Commit Date: 08/17/26

https://github.com/wistaria/MateriAppsInstaller

Stars: 14

Forks: 14

Last Commit Date: 05/24/26

https://github.com/wtmatlock/flanker

Stars: 37

Forks: 9

Last Commit Date: 12/18/24

https://github.com/OpenMD/OpenMD

Stars: 91

Forks: 35

Last Commit Date: 08/08/26

https://github.com/google/uis-rnn

Stars: 1.6k

Forks: 320

Last Commit Date: 09/25/24

https://github.com/leftygray/Cascade_calculations

Stars: 0

Forks: 0

Last Commit Date: 03/29/26

https://github.com/NREL/OpenStudio-server

Stars: 59

Forks: 27

Last Commit Date: 08/19/26

In [ ]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '000b054f839a0eaba1d76e97da534b18df0e41f0', 'tree': '798c2f825ab29c72164485cd199f5a23e2d3029c', 'parent': ['fd6b0cde3513b99336aa9aa892c94bbbb11c8f82'], 'author': 'George Artavanis <ga6@verde.princeton.edu>', 'author_time': 1729186857, 'author_tz': '-0400', 'committer': 'George Artavanis <ga6@verde.princeton.edu>', 'committer_time': 1729186857, 'committer_tz': '-0400', 'message': 'Format with black\n'}


In [ ]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [ ]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,nrel_openstudio-server,000b054f839a0eaba1d76e97da534b18df0e41f0,George Artavanis <ga6@verde.princeton.edu>,1729186857,Format with black\n


In [ ]:
#mode a means append, so you have all your projects in the same file
yournetid='sydlwils'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github